In [ ]:
import os, sys, json, zipfile, hashlib, traceback
from pathlib import Path
run_id = 'V005_N1_N2_CPU_A003'
root = Path('/content/mla_task_runs/V005_N1_N2_CPU_A003')
stages = {'run_id':run_id,'setup':False,'data':False,'cuda':False,'training':False}
assert not (root/'execution_started.json').exists(), 'Attempt already executed; create a new run ID'
root.mkdir(parents=True,exist_ok=True)
bundle = Path('/content/mla_task_bundles/V005_N1_N2_CPU_A003.zip')
with zipfile.ZipFile(bundle) as archive: archive.extractall(root/'source')
sys.path.insert(0,str(root/'source'))
for name in list(sys.modules):
    if name == 'mla_exp' or name.startswith('mla_exp.') or name in ['span_model','sequence_model','llm_pilot','llm_route','hybrid_model','final_evaluation','assessment']: del sys.modules[name]
config = json.loads((root/'source/config.json').read_text())
import span_model
span_model.SETTINGS.update(config)
from mla_exp.common import write_json
if config.get('parent_checkpoint'):
    import shutil
    parent = Path(config['parent_checkpoint'])
    assert parent.is_absolute() and str(parent).startswith('/content/mla_task_runs/')
    assert (parent/'training.pt').exists() and (parent/'model/span_head.json').exists()
    parent_run = parent.parent.parent
    target = Path(config['output_root'])/'runs'/run_id
    target.mkdir(parents=True,exist_ok=False)
    child_checkpoint = target/'checkpoints'/parent.name
    shutil.copytree(parent,child_checkpoint)
    shutil.copytree(parent_run/'best',target/'best')
    shutil.copy2(parent_run/'metadata.json',target/'metadata.json')
    write_json(target/'resume.json',{'checkpoint':str(child_checkpoint),'parent':str(parent)})
write_json(root/'execution_started.json',{'run_id':run_id,'source_sha256':hashlib.sha256(bundle.read_bytes()).hexdigest(),'config':config})
print('CONFIG',json.dumps(config))
stages['setup'] = True


In [ ]:
from mla_exp.data import load_frame, assert_disjoint
from mla_exp import LABELS
assert stages['run_id']==run_id and stages['setup'], 'Setup did not pass'
assert list(LABELS) == ['Ineffective','Adequate','Effective'], LABELS
development = load_frame(config['data_root'], 'development', config)
assert_disjoint(development)
for row in development.sample(min(100,len(development)),random_state=42).to_dict('records'):
    assert ' '.join(row['full_text'][int(row['discourse_start']):int(row['discourse_end'])].split()) == ' '.join(row['discourse_text'].split())
print('DATA_CONTRACT_PASS', len(development), development.groupby('split').group_id.nunique().to_dict())
stages['data'] = True


In [ ]:
assert stages['run_id']==run_id and stages['data'], 'Data contract did not pass'
import torch
torch.set_num_threads(1)
print('CPU_ENSEMBLE: aligned saved validation/calibration predictions only; no CUDA inference')
stages['cuda'] = True


In [ ]:
assert stages['run_id']==run_id and stages['cuda'], 'CUDA contract did not pass'
from final_evaluation import choose_ensemble
result=choose_ensemble(config['source_best'],config['source_b'],config,Path(config['output_root'])/'runs'/run_id)
print('ENSEMBLE_VALIDATION',json.dumps(result))
write_json(root/'status.json',{'status':'model_ready','run_id':run_id})

stages['training'] = True


In [ ]:
assert stages['run_id']==run_id and stages['training'], 'Training did not complete'
from mla_exp.evaluation import evaluate_file
for p in Path(config['output_root']).glob('runs/'+run_id+'/predictions_*.csv.gz'):
    report = p.parent/('evaluation_'+p.name.removeprefix('predictions_').removesuffix('.csv.gz'))/'report.json'
    evaluate_file(p,report,repeats=2000,data_root=config['data_root'])
    print('EVALUATION',report.name)
for p in Path(config['output_root']).glob('runs/*/metrics*.json'):
    print(p.name,p.read_text())

write_json(root/'status.json',{'status':'complete','run_id':run_id})
print('RUN_COMPLETE',run_id)
